# Final Milestone — LLM Comparison and Tool Demo

This notebook covers:
1. Load retrievers and RAG pipeline components
2. Compare two LLMs: `Meta-Llama-3-8B-Instruct` vs `Qwen/Qwen3.5-2B`
3. Run 5 queries through both models with identical retriever and prompt
4. Demonstrate Tavily web search tool on 3 example queries

In [ ]:
import sys
from pathlib import Path

sys.path.insert(0, str(Path.cwd().parent))

from dotenv import load_dotenv
load_dotenv(Path.cwd().parent / ".env")

In [ ]:
from src.bm25 import BM25Retriever
from src.semantic import SemanticRetriever

INDICES = Path.cwd().parent / "indices"

bm25 = BM25Retriever()
bm25.load(str(INDICES / "bm25_index.pkl"))

semantic = SemanticRetriever()
semantic.load(str(INDICES / "faiss_index"))

print(f"BM25 corpus: {len(bm25.corpus):,} products")
print(f"Semantic corpus: {len(semantic.corpus):,} products")

## Step 1.2: LLM Comparison

We compare two models using **identical** retrieval (Hybrid/RRF) and prompt (`strict_citation`):

| Model | Parameters | Family | Provider |
|---|---|---|---|
| `meta-llama/Meta-Llama-3-8B-Instruct` | 8B | Meta Llama 3 | HF Inference API |
| `Qwen/Qwen3.5-2B` | 2B | Alibaba Qwen 3.5 | HF Inference API |

The 4x parameter difference should reveal quality-vs-size tradeoffs in citation accuracy, completeness, and fluency.

In [ ]:
from src.rag_pipeline import RAGPipeline, load_llm

MODEL_A = "meta-llama/Meta-Llama-3-8B-Instruct"
MODEL_B = "Qwen/Qwen3.5-2B"

llm_a = load_llm(model_id=MODEL_A)
llm_b = load_llm(model_id=MODEL_B)

print(f"Model A: {MODEL_A}")
print(f"Model B: {MODEL_B}")

In [ ]:
pipeline_a = RAGPipeline(
    bm25=bm25, semantic=semantic,
    retriever_name="Hybrid", prompt_name="strict_citation",
    llm=llm_a, top_k=5,
)
pipeline_b = RAGPipeline(
    bm25=bm25, semantic=semantic,
    retriever_name="Hybrid", prompt_name="strict_citation",
    llm=llm_b, top_k=5,
)

comparison_queries = [
    "vitamin C serum for brightening",
    "what helps with mild acne and post-acne marks?",
    "gentle face wash that won't irritate sensitive skin",
    "best skincare routine for oily acne-prone teenage skin under $30",
    "what helps with sun damage on fair skin around the eyes",
]

results_comparison = []
for q in comparison_queries:
    print("=" * 80)
    print("QUERY:", q)
    print("=" * 80)

    result_a = pipeline_a.answer(q)
    print("--- Model A (Llama-3-8B) ---")
    print(result_a["answer"])
    print()

    result_b = pipeline_b.answer(q)
    print("--- Model B (Qwen3.5-2B) ---")
    print(result_b["answer"])
    print()

    results_comparison.append({
        "query": q,
        "answer_a": result_a["answer"],
        "answer_b": result_b["answer"],
        "sources_a": result_a["sources"],
        "sources_b": result_b["sources"],
    })

In [ ]:
import json

COMPARISON_OUT = Path.cwd().parent / "data" / "eval_outputs" / "llm_comparison.json"
COMPARISON_OUT.parent.mkdir(parents=True, exist_ok=True)
COMPARISON_OUT.write_text(json.dumps(results_comparison, indent=2))
print(f"Saved {len(results_comparison)} comparison results to {COMPARISON_OUT}")

## Step 2: Tool Integration Demo (Tavily Web Search)

We demonstrate the Tavily web search tool (`src/tools.py`) on 3 queries where the corpus context alone may be insufficient — e.g., queries about current prices, recent product launches, or ingredient safety information that reviews don't cover.

In [ ]:
from src.tools import web_search

tool_queries = [
    "Is retinol safe to use with vitamin C serum?",
    "best drugstore sunscreen 2025 dermatologist recommended",
    "CeraVe moisturizer recall or safety issues",
]

tool_results = []
for q in tool_queries:
    print("QUERY:", q)
    print("-" * 60)

    # RAG answer (from corpus only)
    rag_result = pipeline_a.answer(q)
    print("RAG-only answer (first 300 chars):")
    print(rag_result["answer"][:300])
    print()

    # Web search result
    web_result = web_search.invoke({"query": q})
    print("Tavily web search result (first 500 chars):")
    print(web_result[:500] if web_result else "(no results)")
    print()

    tool_results.append({
        "query": q,
        "rag_answer": rag_result["answer"],
        "web_result": web_result,
    })

TOOL_OUT = Path.cwd().parent / "data" / "eval_outputs" / "tool_demo.json"
TOOL_OUT.write_text(json.dumps(tool_results, indent=2))
print(f"Saved {len(tool_results)} tool demo results to {TOOL_OUT}")